In [ ]:
!pip install -q PyMuPDF chromadb sentence-transformers

import os
import fitz
import chromadb
from sentence_transformers import SentenceTransformer
import shutil
import io
from PIL import Image

PDF_INPUT_DIR = "/kaggle/input/assam-disaster-pdf-knowledge-base2"
DB_SAVE_PATH = "/kaggle/working/embedding_vectors"
IMG_SAVE_PATH = "/kaggle/working/pics"
os.makedirs(DB_SAVE_PATH, exist_ok=True)
os.makedirs(IMG_SAVE_PATH, exist_ok=True)

print(f"PDFs will be read from: {PDF_INPUT_DIR}")
print(f"Database will be saved to: {DB_SAVE_PATH}")
print(f"Extracted images will be saved to: {IMG_SAVE_PATH}")
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face login successful!")
def process_pdfs_to_chroma(pdf_dir, db_path, img_dir):
    print("Loading text embedding model (google/embeddinggemma-300m)...")
    text_embed_model = SentenceTransformer('google/embeddinggemma-300m', device="cuda")

    client = chromadb.PersistentClient(path=db_path)

    collection_name = "assam_disasters_multimodal"

    if collection_name in [c.name for c in client.list_collections()]:
        print(f"Collection '{collection_name}' exists.")
        client.delete_collection(name=collection_name)

    collection = client.create_collection(name=collection_name)
    print(f"ChromaDB collection '{collection_name}' created at {db_path}")

    total_pdfs_processed = 0
    total_chunks_indexed = 0

    print(f"Starting to process PDFs in {pdf_dir}...")

    for root, dirs, files in os.walk(pdf_dir):
        for file in files:
            if file.lower().endswith(".pdf"):
                pdf_file_path = os.path.join(root, file)
                print(f"\n--- Processing: {file} ---")

                try:
                    doc = fitz.open(pdf_file_path)
                    total_pdfs_processed += 1

                    batch_docs = []
                    batch_metadatas = []
                    batch_ids = []

                    for page_num in range(len(doc)):
                        page = doc.load_page(page_num)
                        page_text = page.get_text("text")
                        saved_image_paths = []
                        image_list = page.get_images(full=True)

                        for img_index, img in enumerate(image_list):
                            xref = img[0]
                            base_image = doc.extract_image(xref)
                            image_bytes = base_image["image"]

                            img_filename = f"{os.path.splitext(file)[0]}_p{page_num+1}_{img_index}.png"
                            img_save_path = os.path.join(img_dir, img_filename)

                            try:
                                pil_image = Image.open(io.BytesIO(image_bytes))
                                pil_image.save(img_save_path)
                                saved_image_paths.append(img_save_path)
                            except Exception as e:
                                print(f"Warning: Could not save image on page {page_num+1} from {file}: {e}")
                        if page_text.strip():
                            chunk_id = f"{file}_p{page_num+1}"

                            metadata = {
                                "source_pdf": file,
                                "page_number": page_num + 1,
                                "image_paths": ",".join(saved_image_paths)
                            }

                            batch_docs.append(page_text)
                            batch_metadatas.append(metadata)
                            batch_ids.append(chunk_id)
                            total_chunks_indexed += 1
                    if batch_docs:
                        print(f"  > Adding {len(batch_docs)} pages to database...")
                        batch_embeddings = text_embed_model.encode(batch_docs).tolist()

                        collection.add(
                            embeddings=batch_embeddings,
                            documents=batch_docs,
                            metadatas=batch_metadatas,
                            ids=batch_ids
                        )

                    doc.close()

                except Exception as e:
                    print(f"ERROR: Could not process {file}: {e}")

    print("\nIndexing Complete!")
    print(f"Total PDFs processed: {total_pdfs_processed}")
    print(f"Total pages (chunks) indexed: {total_chunks_indexed}")
    print(f"Database is saved in: {db_path}")
    print(f"All images are saved in: {img_dir}")
process_pdfs_to_chroma(PDF_INPUT_DIR, DB_SAVE_PATH, IMG_SAVE_PATH)
print("Verifying database...")
try:
    client = chromadb.PersistentClient(path=DB_SAVE_PATH)
    collection = client.get_collection(name="assam_disasters_multimodal")
    count = collection.count()
    print(f"Success! Database loaded.")
    print(f"Total items in collection: {count}")

    print("\nVerifying images...")
    image_files = os.listdir(IMG_SAVE_PATH)
    print(f"Total images extracted: {len(image_files)}")
    if image_files:
        print(f"Example image: {image_files[0]}")
except Exception as e:
    print(f"Verification failed: {e}")